# SR Equations in Physical Form

The SR equations are discovered and optimized in **normalized** (z-scored) space,
where each feature is standardized to zero mean and unit variance using training-set
statistics. This notebook converts each optimized equation into **physical form** —
expressed in terms of the original un-normalized variables — by absorbing the
normalization constants into the equation coefficients.

This makes the equations interpretable in physical units and directly comparable
to theory (e.g., buoyancy thresholds in K, relative humidity fractions).

In [ ]:
import os
import json
import pickle
import numpy as np
import sympy as sp

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR = CONFIGS['filepaths']['splits']
MODELSDIR = CONFIGS['filepaths']['models']
SRMODELS  = CONFIGS['experiments']['sr']['optimizedeqs']

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)

regpath = os.path.join(MODELSDIR,'sr','optimized_equations.pkl')
with open(regpath,'rb') as f:
    REGISTRY = pickle.load(f)

ORDER = [name for name in SRMODELS if name in REGISTRY]
LABELS = {name:SRMODELS[name]['description'] for name in ORDER}

TPMEAN = STATS['tp_mean']
TPSTD  = STATS['tp_std']
ZMIN   = (0.0 - TPMEAN) / TPSTD

print('Normalization statistics (training set):')
print(f'  tp: mean={TPMEAN:.6f}, std={TPSTD:.6f}, zmin={ZMIN:.4f}')
for var in ['rh','thetae','thetaestar','bl','shf','lhf']:
    m = STATS.get(f'{var}_mean',None)
    s = STATS.get(f'{var}_std',None)
    if m is not None:
        print(f'  {var}: mean={m:.6f}, std={s:.6f}')
print()
print('Optimized equations (normalized form):')
for name in ORDER:
    entry = REGISTRY[name]
    cstr = ', '.join(f'{k}={v:.4f}' for k,v in entry['constants'].items())
    print(f'  {LABELS[name]}: {entry["form"]}  [{cstr}]')

## Normalization transform

Each feature $x$ enters the SR equation as its z-scored form:

$$\hat{x} = \frac{x - \mu_x}{\sigma_x}$$

The target is $\log(1 + P)$, z-scored:

$$z = \frac{\log(1 + P) - \mu_t}{\sigma_t}$$

The prediction pipeline is:
1. Compute `raw = f(normalized features)` using the SR equation
2. Clamp: `z = zmin + max(raw, 0)` where `zmin = -mean/std`
3. Back-transform: `P = exp(z * std + mean) - 1`

To express in physical form, substitute $\hat{x} = (x - \mu_x) / \sigma_x$ into
the equation and simplify. The result is a function of physical variables with
modified coefficients that absorb the normalization.

In [ ]:
SRSYMPY = {
    'cube':lambda x:x**3,
    'square':lambda x:x**2,
    'neg':lambda x:-x,
    'sqrt':sp.sqrt,
    'exp':sp.exp,
    'log':sp.log,
    'abs':sp.Abs,
    'sin':sp.sin,
    'cos':sp.cos,
    'max':sp.Max,
    'min':sp.Min}

FEATUREVARS = ['rh','thetae','thetaestar','bl','shf','lhf','lf']

normsyms = {v:sp.Symbol(f'{v}_norm') for v in FEATUREVARS}
physsyms = {v:sp.Symbol(v) for v in FEATUREVARS}

def norm_to_phys_sub(var):
    m = STATS.get(f'{var}_mean')
    s = STATS.get(f'{var}_std')
    if m is None or s is None:
        return normsyms[var],physsyms[var]
    return normsyms[var],(physsyms[var] - sp.Rational(m).limit_denominator(10**8)) / sp.Rational(s).limit_denominator(10**8)

## SR-BL: Physical form

Normalized: $\text{raw} = (\hat{B}_L + a)^3 + b$

Substituting $\hat{B}_L = (B_L - \mu_{BL}) / \sigma_{BL}$:

$$\text{raw} = \left(\frac{B_L - \mu_{BL}}{\sigma_{BL}} + a\right)^3 + b = \left(\frac{B_L - (\mu_{BL} - a \cdot \sigma_{BL})}{\sigma_{BL}}\right)^3 + b$$

This reveals the **physical threshold**: precipitation onset occurs at
$B_L \approx \mu_{BL} - a \cdot \sigma_{BL}$.

In [ ]:
if 'sr_bl_eq' in REGISTRY:
    entry = REGISTRY['sr_bl_eq']
    c = entry['constants']
    blmean = STATS['bl_mean']
    blstd  = STATS['bl_std']

    threshold_bl = blmean - c['a'] * blstd
    scale_bl = 1.0 / blstd

    print('SR-BL in normalized form:')
    print(f'  raw = (bl_norm + {c["a"]:.4f})^3 + {c["b"]:.4f}')
    print()
    print('SR-BL in physical form:')
    print(f'  raw = ((B_L - {threshold_bl:.4f}) / {blstd:.4f})^3 + {c["b"]:.4f}')
    print()
    print(f'Physical interpretation:')
    print(f'  Onset threshold: B_L = {threshold_bl:.4f} (normalized: bl_norm = {-c["a"]:.4f})')
    print(f'  bl_mean = {blmean:.4f}, bl_std = {blstd:.4f}')
    print(f'  Threshold = mean - a*std = {blmean:.4f} - {c["a"]:.4f}*{blstd:.4f} = {threshold_bl:.4f}')
    print(f'  Sensitivity: 1/sigma_BL = {scale_bl:.4f} per unit B_L')
else:
    print('SR-BL not in registry — run optimizer first.')

## SR-ATM: Physical form

Normalized: $\text{raw} = a \cdot \left(\max\left(\hat{\text{RH}},\; b \cdot \hat{\theta}_e - c \cdot \hat{\theta}_e^* - d\right)\right)^3$

Substituting normalized features:

$$\hat{\text{RH}} = \frac{\text{RH} - \mu_{\text{RH}}}{\sigma_{\text{RH}}}, \quad
\hat{\theta}_e = \frac{\theta_e - \mu_{\theta_e}}{\sigma_{\theta_e}}, \quad
\hat{\theta}_e^* = \frac{\theta_e^* - \mu_{\theta_e^*}}{\sigma_{\theta_e^*}}$$

The moisture pathway becomes:
$$\frac{\text{RH} - \mu_{\text{RH}}}{\sigma_{\text{RH}}}$$

The instability pathway becomes:
$$\frac{b}{\sigma_{\theta_e}} \cdot \theta_e - \frac{c}{\sigma_{\theta_e^*}} \cdot \theta_e^*
- \left(d + \frac{b \cdot \mu_{\theta_e}}{\sigma_{\theta_e}} - \frac{c \cdot \mu_{\theta_e^*}}{\sigma_{\theta_e^*}}\right)$$

This reveals the physical balance between buoyancy ($\theta_e$) and stability
($\theta_e^*$) in controlling the instability pathway.

In [ ]:
if 'sr_atm_eq' in REGISTRY:
    entry = REGISTRY['sr_atm_eq']
    c = entry['constants']

    rhmean = STATS['rh_mean']
    rhstd  = STATS['rh_std']
    temean = STATS['thetae_mean']
    testd  = STATS['thetae_std']
    tesmean = STATS['thetaestar_mean']
    tesstd  = STATS['thetaestar_std']

    print('SR-ATM in normalized form:')
    cstr = ', '.join(f'{k}={v:.4f}' for k,v in c.items())
    print(f'  raw = {entry["form"]}  [{cstr}]')
    print()

    ca = c.get('a',c.get('a'))
    cb = c.get('b',c.get('d',1.0))
    cc = c.get('c',c.get('e',1.0))
    cd = c.get('d',c.get('f',0.0))

    phys_rh_scale = 1.0 / rhstd
    phys_rh_offset = rhmean

    phys_te_coef  = cb / testd
    phys_tes_coef = cc / tesstd
    phys_offset   = cd + cb * temean / testd - cc * tesmean / tesstd

    print('SR-ATM in physical form:')
    print(f'  Moisture pathway:    (RH - {phys_rh_offset:.4f}) / {rhstd:.4f}')
    print(f'  Instability pathway: {phys_te_coef:.4f} * thetae - {phys_tes_coef:.4f} * thetaestar - {phys_offset:.4f}')
    print(f'  raw = {ca:.4f} * cube(max(moisture, instability))')
    print()
    print('Physical coefficients:')
    print(f'  a (overall scale):          {ca:.4f}')
    print(f'  RH scale (1/sigma_RH):      {phys_rh_scale:.6f}')
    print(f'  RH offset (mu_RH):          {phys_rh_offset:.4f}')
    print(f'  thetae coef (b/sigma_te):   {phys_te_coef:.6f}')
    print(f'  thetaestar coef (c/sigma_tes): {phys_tes_coef:.6f}')
    print(f'  instab. offset:             {phys_offset:.4f}')
    print()

    ratio = phys_te_coef / phys_tes_coef
    print('Physical interpretation:')
    print(f'  thetae/thetaestar sensitivity ratio: {ratio:.4f}')
    print(f'  For the instability pathway to dominate moisture, need:')
    print(f'    {phys_te_coef:.4f}*thetae - {phys_tes_coef:.4f}*thetaestar > (RH - {phys_rh_offset:.4f})/{rhstd:.4f} + {phys_offset:.4f}')
else:
    print('SR-ATM not in registry — run optimizer first.')

## Summary table

Physical-form coefficients for all optimized SR equations, with the normalization
constants absorbed. These can be written directly in the paper as equations in
physical units.

In [ ]:
import pandas as pd

print('='*80)
print('PHYSICAL-FORM EQUATIONS')
print('='*80)
print()
print('Prediction pipeline:')
print(f'  z = {ZMIN:.4f} + max(raw, 0)')
print(f'  P = exp(z * {TPSTD:.6f} + {TPMEAN:.6f}) - 1  [mm]')
print()

for name in ORDER:
    entry = REGISTRY[name]
    print(f'--- {LABELS[name]} ---')
    print(f'  Normalized:  raw = {entry["form"]}')
    print(f'  Constants:   {entry["constants"]}')
    print()

## Regime boundaries in physical space

The `max(RH, instability)` in SR-ATM creates a regime boundary. In normalized
space this is where `rh_norm = b*thetae_norm - c*thetaestar_norm - d`.
Converting to physical space tells us which thermodynamic conditions favor
the moisture vs. instability control on precipitation.

In [ ]:
if 'sr_atm_eq' in REGISTRY:
    c = REGISTRY['sr_atm_eq']['constants']

    ca = c.get('a',c.get('a'))
    cb = c.get('b',c.get('d',1.0))
    cc = c.get('c',c.get('e',1.0))
    cd = c.get('d',c.get('f',0.0))

    rhmean = STATS['rh_mean']
    rhstd  = STATS['rh_std']
    temean = STATS['thetae_mean']
    testd  = STATS['thetae_std']
    tesmean = STATS['thetaestar_mean']
    tesstd  = STATS['thetaestar_std']

    print('Regime boundary in normalized space:')
    print(f'  rh_norm = {cb:.4f}*thetae_norm - {cc:.4f}*thetaestar_norm - {cd:.4f}')
    print()
    print('Regime boundary in physical space:')
    print(f'  (RH - {rhmean:.4f}) / {rhstd:.4f} = {cb:.4f}*(thetae - {temean:.4f})/{testd:.4f}'
          f' - {cc:.4f}*(thetaestar - {tesmean:.4f})/{tesstd:.4f} - {cd:.4f}')
    print()
    print('Simplified:')
    A_rh = 1.0 / rhstd
    A_te = cb / testd
    A_tes = cc / tesstd
    A_const = cd + cb * temean / testd - cc * tesmean / tesstd - rhmean / rhstd
    print(f'  {A_rh:.6f}*RH = {A_te:.6f}*thetae - {A_tes:.6f}*thetaestar - {A_const:.4f}')
    print()
    print('Moisture dominates when: RH is large relative to buoyancy-stability imbalance')
    print('Instability dominates when: buoyancy exceeds stability enough to overwhelm moisture')
else:
    print('SR-ATM not in registry.')